# T12 — Minimal Y-factor calibration of D5 data

**What this is.** The smallest complete version of the T09 calibration chain,
written to be shared with collaborators. From the raw D5 files alone it gives:

1. the three-state Y factor $T^* = T_\mathrm{load} + T_\mathrm{NS}\,
   \dfrac{P_\mathrm{ant} - P_\mathrm{load}}{P_\mathrm{NS} - P_\mathrm{load}}$;
2. $T^*$ corrected for the antenna S11;
3. $T^*$ corrected for the antenna, ambient-load and receiver S11 together
   (T09 § 8).

The load and noise-source spectra and the S11 are paired with each sky visit by
**nearest neighbour in time**. The load temperature is the thermistor reading
(`tempctrl_load.T_now`). The noise-source excess $T_\mathrm{NS}$ is an assumed
value, `d5_yfactor.T_NS`: nameplate 917 K, **not measured**.

**What it does not do.** No RFI flagging, no smoothing, no interpolation in time,
no maximum-age cut on the calibrators or S11. The only rows excluded are dropped
integrations (all-zero rows), in § 2, where they are counted. Everything else,
including RFI and the 1 MHz comb burst in the 02:53 MDT noise-source visit,
passes straight through.

**Files to share:** this notebook and `d5_yfactor.py` next to it. It needs
`numpy`, `h5py`, `matplotlib`, `pandas` (tables only) and `cmt_vna`. It reads
the D5 correlator files (filtered or raw), the 40 S11 files and the two files in
`field_cal_data_2026/cal_materials/`. It writes nothing.

**Status: diagnostic.** No noise-wave, loss or beam correction. The kelvin scale
rests on the assumed $T_\mathrm{NS}$.

In [ ]:
# Provenance: the eigsep_analysis record where it exists, else versions.
try:
    from eigsep_analysis import provenance
    provenance.show()
except ImportError:
    import sys, h5py, numpy, cmt_vna
    print(sys.version.split()[0], 'numpy', numpy.__version__,
          'h5py', h5py.__version__, 'cmt_vna', cmt_vna.__version__)

## 0. Settings

**This copy** (Aaron's, imported 2026-09-25 from Christian's standalone bundle, itself from `christianhbye/eigsep_talks` @ `5bf4b1b`) differs from the original only here: the paths resolve to the checkout — `marjum-2026-07/data/` (filtered correlator files), `marjum-2026-07/data/s11/` and `eigsep_observing`'s package data for the two lab files. `CORR_DIR` can be the filtered set
or the raw set: both store the autos as `data/<key>`. In phase C (from Jul 15)
key `"4"` is the suspended (box-air) antenna, the input the calibration switch
serves. The display settings change the plots only, never the arrays.

In [ ]:
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import matplotlib.colors as mcolors
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import d5_yfactor as yf

# Resolved from the checkout, not hard-coded: the campaign archive for the
# correlator and S11 files, eigsep_observing's package data for the lab files.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'marjum-2026-07/data').is_dir())
CORR_DIR = ROOT / 'marjum-2026-07' / 'data'
S11_DIR = CORR_DIR / 's11'
CAL_DIR = ROOT / 'eigsep_observing' / 'src' / 'eigsep_observing' / 'data'
INTERNAL_OSL = CAL_DIR / 'vna_internal_osl_20260629T090825Z.npz'
SWITCH_SPARAMS = CAL_DIR / 'switch_sparams.npz'

TZ = ZoneInfo('America/Denver')
START = datetime(2026, 7, 16, 21, 30, tzinfo=TZ)   # the clean night of 16/17 Jul
STOP = datetime(2026, 7, 17, 7, 55, tzinfo=TZ)
KEY = '4'

# Display only
PLOT_FREQ_MHZ = (50, 250)
PLOT_BAND_MHZ = (50, 87)   # band averaged for the time-series plots

print(f'{START:%Y-%m-%d %H:%M} to {STOP:%Y-%m-%d %H:%M} MDT, key {KEY!r}')
print(f'T_NS = {yf.T_NS:.1f} K (assumed; change d5_yfactor.T_NS)')

In [ ]:
# Plot helpers.
def as_num(t):
    # unix seconds -> matplotlib date numbers
    return mdates.date2num((np.asarray(t) * 1e6).astype('datetime64[us]'))

def time_axis(axis, label=True):
    axis.set_major_locator(mdates.HourLocator(interval=2, tz=TZ))
    axis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=TZ))
    if label:
        axis.set_label_text('local time (MDT)')

def band(freqs, lo_hi=PLOT_BAND_MHZ):
    return (freqs >= lo_hi[0]) & (freqs < lo_hi[1])

def waterfall(ax, visits, values, freqs, label, norm=None):
    # Each visit is drawn over its own [t_start, t_stop]; the time between
    # visits stays blank. Nonpositive values are blank on a log scale.
    edges = as_num(np.column_stack([visits['t_start'],
                                    visits['t_stop']]).ravel())
    z = np.full((2 * len(values) - 1, values.shape[1]), np.nan)
    z[::2] = values
    df = freqs[1] - freqs[0]
    f_edges = np.r_[freqs - df / 2, freqs[-1] + df / 2]
    if norm is None:
        v = values[:, band(freqs, PLOT_FREQ_MHZ)]
        v = v[np.isfinite(v) & (v > 0)]
        norm = mcolors.LogNorm(*np.percentile(v, [1, 99]))
    mesh = ax.pcolormesh(f_edges, edges, z, norm=norm, shading='flat',
                         rasterized=True)
    ax.set(xlim=PLOT_FREQ_MHZ, ylim=(edges[-1], edges[0]),
           xlabel='frequency (MHz)')
    time_axis(ax.yaxis)
    ax.figure.colorbar(mesh, ax=ax, label=label)

## 1. Load the correlator data

`yf.load_corr` reads every integration of `KEY` in the window, with:

- **time.** The header time, unless that file's clock was stale (its last
  header time more than 1 h from the filename stamp). Then the time counts back
  from the filename stamp in steps of the integration time. About 12 % of D5
  files are stale; their count in this window is printed.
- **switch state.** `rfswitch`, or `MISSING` where none was recorded.
- **load temperature.** `tempctrl_load.T_now`, in K.

Nothing is dropped here. The counts of all-zero rows (dropped integrations) and
of rows with negative powers (int32 accumulator wraps in strong narrow lines)
are printed so they are on the record.

In [ ]:
d = yf.load_corr(CORR_DIR, START.timestamp(), STOP.timestamp(), KEY)
freqs = d['freqs']
zero = np.all(d['spec'] == 0, axis=1)
wrapped = d['spec'] < 0

print(f"{len(d['time']):,} integrations from {len(set(d['file']))} files; "
      f"{len(freqs)} channels, {freqs[0]:.3f}–{freqs[-1]:.3f} MHz")
print(f"rows timed from the filename (stale clock): "
      f"{d['time_from_filename'].sum()}")
print('switch states:', dict(pd.Series(d['state']).value_counts()))
print(f"all-zero rows: {zero.sum()}; rows with a negative (wrapped) value: "
      f"{wrapped.any(axis=1).sum()}, in {wrapped.any(axis=0).sum()} channels")
print(f"rows with no load thermistor reading: {np.isnan(d['t_load']).sum()}")

In [ ]:
states = [yf.SKY, yf.NOISE, yf.LOAD] + sorted(
    set(d['state']) - {yf.SKY, yf.NOISE, yf.LOAD})
fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(12, 6), sharex=True,
                               height_ratios=(2, 1))
y = np.array([states.index(s) for s in d['state']])
ax0.scatter(as_num(d['time']), y, s=1, rasterized=True)
ax0.set_yticks(range(len(states)), states)
ax0.set_title('switch state of every integration')
ax1.plot(as_num(d['time']), d['t_load'], '.', ms=1)
ax1.set_ylabel('load thermistor (K)')
time_axis(ax1.xaxis)
fig.tight_layout()

## 2. Average each switch visit

A **visit** is a contiguous run of one switch state (a new one also starts at a
time gap over `yf.MAX_GAP_S` = 5 s). Each visit is reduced to one spectrum, the
**median** over its rows: the wrapped values from § 1 sit in a few rows of a
handful of RFI channels, and a mean would carry them into the average. Pass
`stat=np.mean` to see the difference. All-zero rows are excluded here, and that
is the only exclusion in this notebook. Each visit's load temperature is the
mean thermistor reading over its rows.

In [ ]:
visits = yf.average_blocks(d, use=~zero)
sky, noise, load = (visits[s] for s in (yf.SKY, yf.NOISE, yf.LOAD))
for s, v in visits.items():
    print(f"{s}: {len(v['t'])} visits, {v['n'].min()}–{v['n'].max()} "
          f"rows each")

In [ ]:
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(15, 5))
b = band(freqs)
for s, v in visits.items():
    ax0.semilogy(as_num(v['t']), v['spec'][:, b].mean(axis=1), '.', label=s)
ax0.set(ylabel=f'mean power {PLOT_BAND_MHZ[0]}–{PLOT_BAND_MHZ[1]} MHz '
        '(counts)', title='every visit')
time_axis(ax0.xaxis)
ax0.legend()
mid = 0.5 * (START.timestamp() + STOP.timestamp())
for s, v in visits.items():
    i = np.argmin(np.abs(v['t'] - mid))
    t = datetime.fromtimestamp(v['t'][i], TZ)
    ax1.semilogy(freqs, np.where(v['spec'][i] > 0, v['spec'][i], np.nan),
                 lw=1, label=f'{s} {t:%H:%M}')
ax1.set(xlim=PLOT_FREQ_MHZ, xlabel='frequency (MHz)',
        ylabel='power (counts)', title='one visit of each state')
ax1.legend()
fig.tight_layout()

## 3. Nearest load and noise-source visit for each sky visit

Each sky visit takes the spectrum and thermistor reading of the load visit
nearest to it in time, and the spectrum of the nearest noise-source visit. No
limit is put on how far away that visit may be; the ages are plotted instead.

In [ ]:
i_load, age_load = yf.nearest(sky['t'], load['t'])
i_noise, age_noise = yf.nearest(sky['t'], noise['t'])
P_sky = sky['spec']
P_load = load['spec'][i_load]
P_noise = noise['spec'][i_noise]
T_load = load['t_load'][i_load]
print(f"largest age: load {age_load.max() / 60:.1f} min, "
      f"noise source {age_noise.max() / 60:.1f} min")
print(f"T_load {np.nanmin(T_load):.2f}–{np.nanmax(T_load):.2f} K")

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(as_num(sky['t']), age_load / 60, 'o', label=yf.LOAD)
ax.plot(as_num(sky['t']), age_noise / 60, 's', mfc='none', label=yf.NOISE)
ax.set(ylabel='age (min)', title='time from each sky visit to the visit it uses')
time_axis(ax.xaxis)
ax.legend()
fig.tight_layout()

## 4. Three-state Y factor

$T^* = T_\mathrm{load} + T_\mathrm{NS}\,(P_\mathrm{ant} - P_\mathrm{load})
/(P_\mathrm{NS} - P_\mathrm{load})$, channel by channel, with $T_\mathrm{NS}$
the assumed excess of the noise-source state over the load state. This is the
uncalibrated antenna temperature, at the receiver input. The short bright
dashes near 137 MHz are ORBCOMM satellite passes during single sky visits.

In [ ]:
T_star = yf.tant_star(P_sky, P_noise, P_load, T_load[:, None])

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(16, 6))
waterfall(ax0, sky, T_star, freqs, r'$T^*$ (K)')
ax0.set_title(r'$T^*$, every sky visit')
pick = np.linspace(0, len(sky['t']) - 1, 6).astype(int)
for i in pick:
    t = datetime.fromtimestamp(sky['t'][i], TZ)
    ax1.semilogy(freqs, np.where(T_star[i] > 0, T_star[i], np.nan), lw=1,
                 label=f'{t:%H:%M}')
ax1.set(xlim=PLOT_FREQ_MHZ, xlabel='frequency (MHz)', ylabel=r'$T^*$ (K)',
        title='six sky visits')
ax1.legend()
fig.tight_layout()

## 5. S11 at plane P

`yf.calibrate_s11` takes every raw S11 file to plane P, the receiver input,
with the chain of `calibrate_field_s11.py` (EIGSEP/data-analysis):

1. The VNA error network is solved from each sweep's internal open/short/load
   traces against their lab characterization (`INTERNAL_OSL`) and removed.
2. The lab-measured switch paths (`SWITCH_SPARAMS`) are then de-embedded
   (VNA leg) and embedded (RF leg).

For the antenna this Γ includes the balun and the feed coax; it is not the
free-space antenna of the simulations.

Some sweeps are zero-filled (cmt_vna issue #54). A zero-filled trace is not
calibrated. A sweep whose own internal OSL is zero-filled borrows the valid one
nearest in time, as the team's script does; pass `borrow_osl=False` to leave
those sweeps out instead. The table shows what happened to each sweep in the
window.

In [ ]:
freqs_s11, s11, log = yf.calibrate_s11(S11_DIR, INTERNAL_OSL, SWITCH_SPARAMS)
log = pd.DataFrame(log)
log['time'] = pd.to_datetime(log.t, unit='s', utc=True).dt.tz_convert(TZ)
log['OSL offset (min)'] = ((log.t - log.osl_t) / 60).round(1)
print(pd.crosstab(log.dut, log.status).to_string(), '\n')
# One row per sweep file in the window.
inside = log[(log.t >= START.timestamp()) & (log.t < STOP.timestamp())
             & log.dut.isin(['ant', 'amb', 'rec'])].copy()
inside['sweep (MDT)'] = inside.time.dt.strftime('%m-%d %H:%M')
table = inside.pivot_table(index=['sweep (MDT)', 'file'], columns='dut',
                           values='status', aggfunc='first', fill_value='')
table['OSL offset (min)'] = inside.groupby(
    ['sweep (MDT)', 'file'])['OSL offset (min)'].first()
table

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for ax, dut in zip(axes, ('ant', 'amb', 'rec')):
    s = s11[dut]
    keep = (s['t'] >= START.timestamp()) & (s['t'] < STOP.timestamp())
    for g in s['gamma'][keep]:
        ax.plot(freqs_s11 / 1e6, 20 * np.log10(np.abs(g)), lw=0.8)
    ax.set(xlim=PLOT_FREQ_MHZ, xlabel='frequency (MHz)',
           title=f'{dut}: {keep.sum()} sweeps in the window')
axes[0].set_ylabel(r'$|\Gamma|$ at P (dB)')
fig.tight_layout()

## 6. The two S11 corrections

Each sky visit takes the antenna, ambient-load and receiver sweeps nearest to it
in time, again with no age limit (ages plotted). The S11 grid (1000 points,
1–250 MHz) is put on the correlator channels by linear interpolation of the
real and imaginary parts; channels below 1 MHz are NaN.

- **Antenna S11:** $T^*/(1-|\Gamma_\mathrm{ant}|^2)$, treating the receiver and
  load as matched.
- **Antenna + receiver S11:** with $M_s = (1-|\Gamma_s|^2)/|1-\Gamma_s
  \Gamma_\mathrm{rec}|^2$,
  $T = [T^* - (1-M_\mathrm{load})\,T_\mathrm{load}]/M_\mathrm{ant}$. It
  assumes equal gain on every switch path, no receiver noise waves, and
  $T_\mathrm{NS}$ as the excess delivered to the receiver.

In [ ]:
gam, age_s11 = {}, {}
for dut in ('ant', 'amb', 'rec'):
    i, age_s11[dut] = yf.nearest(sky['t'], s11[dut]['t'])
    gam[dut] = yf.s11_to_channels(s11[dut]['gamma'][i], freqs_s11 / 1e6,
                                  freqs)
T_ant_s11 = yf.correct_antenna_s11(T_star, gam['ant'])
T_rec_s11 = yf.correct_receiver_s11(T_star, T_load[:, None], gam['ant'],
                                    gam['amb'], gam['rec'])
print('largest S11 age (min):',
      {k: round(v.max() / 60, 1) for k, v in age_s11.items()})

fig, ax = plt.subplots(figsize=(12, 3.5))
for dut, m in zip(age_s11, 'os^'):
    ax.plot(as_num(sky['t']), age_s11[dut] / 60, m, mfc='none', label=dut)
ax.set(ylabel='age (min)', title='time from each sky visit to the S11 sweep it uses')
time_axis(ax.xaxis)
ax.legend()
fig.tight_layout()

In [ ]:
stages = [(T_star, r'$T^*$'),
          (T_ant_s11, '+ antenna S11'),
          (T_rec_s11, '+ antenna and receiver S11')]
fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(12, 8), sharex=True,
                               height_ratios=(2, 1))
f = band(freqs, PLOT_FREQ_MHZ)
for arr, label in stages:
    med = np.median(arr[:, f], axis=0)
    ax0.semilogy(freqs[f], np.where(med > 0, med, np.nan), lw=1, label=label)
    ax1.plot(freqs[f], np.median(arr[:, f] / T_star[:, f], axis=0), lw=1)
ax0.set(ylabel='temperature (K)', title='median over the sky visits')
ax0.legend()
ax1.set(xlim=PLOT_FREQ_MHZ, xlabel='frequency (MHz)',
        ylabel=r'ratio to $T^*$')
fig.tight_layout()

fig, ax = plt.subplots(figsize=(8, 6))
waterfall(ax, sky, T_rec_s11, freqs, 'temperature (K)')
ax.set_title('+ antenna and receiver S11, every sky visit')
fig.tight_layout()

## 7. What you have

All arrays are on the native correlator channels `freqs` (MHz), one row per sky
visit; `sky['t']`, `sky['t_start']` and `sky['t_stop']` are unix times.

| Array | Contents |
|---|---|
| `P_sky`, `P_load`, `P_noise` | Visit spectra (counts): the sky visit, and its nearest load and noise-source visits |
| `T_load` | Thermistor temperature of the nearest load visit (K) |
| `T_star` | Three-state Y factor (K) |
| `gam['ant']`, `gam['amb']`, `gam['rec']` | Nearest-in-time Γ at plane P |
| `T_ant_s11`, `T_rec_s11` | `T_star` with the antenna, and the antenna + receiver, S11 corrections |
| `age_load`, `age_noise`, `age_s11` | Time from each sky visit to what it was paired with (s) |

To change the time resolution of the sky, pass your own rows to
`yf.tant_star`: it works on any arrays that broadcast.